# 01 — Tabular Data & First-Principles Statistical Primitives

This notebook explores the fundamentals of tabular data representation, ingestion pipelines, DataFrame-to-array transformations, and first-principles algorithmic calculations of summary statistics on marketing mix advertising data.

## 1. Tabular Data Representation and Ingestion

Tabular data organizes empirical observations into a 2-dimensional grid:
- **Rows (Records / Samples):** Individual observations or entity instances.
- **Columns (Features / Fields):** Attributes measured across observations.

In high-performance computing, tabular data can be represented as:
1. High-level column-oriented containers (`pandas.DataFrame`)
2. Homogeneous numerical tensors (`numpy.ndarray`)
3. Row-major nested primitives (`list[list[float]]`)

In [ ]:
import csv
import math
import time
from pathlib import Path

# Locate dataset path relative to repository
DATA_PATH = Path("../data/advertising_benchmark.csv")
assert DATA_PATH.is_file(), f"Dataset not found at {DATA_PATH}"
print(f"Dataset verified: {DATA_PATH.resolve()}")

## 2. Ingesting Tabular Data and Converting to List of Lists

We read `advertising_benchmark.csv` and convert it into a pure Python row-major list of lists (`advertising_lst`) containing `[tv, radio, newspaper, sales]`. This eliminates high-level library dependencies and allows us to study algorithmic operations from first principles.

In [ ]:
def load_tabular_csv(file_path: Path) -> tuple[list[str], list[list[float]]]:
    """Load a CSV file into a header list and a list of numeric row records."""
    headers = []
    rows = []
    with file_path.open(encoding="utf-8") as f:
        reader = csv.reader(f)
        headers = next(reader)
        for line_no, row in enumerate(reader, start=2):
            if not row:
                continue
            # Row structure: [id, tv, radio, newspaper, sales]
            # We extract numerical features: tv, radio, newspaper, sales
            try:
                numeric_row = [float(val) for val in row[1:]]
                rows.append(numeric_row)
            except ValueError as err:
                raise ValueError(f"Malformed numerical entry at line {line_no}: {row}") from err
    return headers[1:], rows


feature_names, advertising_lst = load_tabular_csv(DATA_PATH)
print(f"Features: {feature_names}")
print(f"Total observations loaded: {len(advertising_lst)}")
print("First 5 records [tv, radio, newspaper, sales]:")
for idx, r in enumerate(advertising_lst[:5], start=1):
    print(f"  Row {idx}: {r}")

## 3. First-Principles Summary Metrics

We implement the mathematical primitives directly on the list-of-lists structure:
1. **Column Sum:** $\sum_{i=1}^N x_{i, c}$
2. **Bounded Slice Sum:** $\sum_{i=	ext{start}}^{	ext{end}} x_{i, c}$
3. **Minimum & Maximum:** $\min(X_c), \max(X_c)$
4. **Arithmetic Mean:** $ar{x} = rac{1}{N} \sum_{i=1}^N x_i$
5. **Median:**
   - For sorted sequence $X'$ of length $N$:
   - If $N$ is odd: $	ext{Median} = X'[\lfloor N/2 floor]$
   - If $N$ is even: $	ext{Median} = rac{X'[N/2 - 1] + X'[N/2]}{2}$
6. **Budget Contribution Percentage:**
   $$	ext{Share}_c = rac{\sum_{i=1}^N x_{i, c}}{\sum_{k \in 	ext{channels}} \sum_{i=1}^N x_{i, k}} 	imes 100\%$$

In [ ]:
def calculate_column_sum(matrix: list[list[float]], col_idx: int) -> float:
    """Compute the exact summation of values in a specific column."""
    return sum(row[col_idx] for row in matrix)


def calculate_slice_sum(
    matrix: list[list[float]], col_idx: int, start_idx: int, end_idx: int
) -> float:
    """Compute column sum within row slice [start_idx, end_idx)."""
    return sum(row[col_idx] for row in matrix[start_idx:end_idx])


def calculate_column_min_max(matrix: list[list[float]], col_idx: int) -> tuple[float, float]:
    """Return the minimum and maximum values of a column."""
    col_values = [row[col_idx] for row in matrix]
    return min(col_values), max(col_values)


def calculate_column_mean(matrix: list[list[float]], col_idx: int) -> float:
    """Compute the arithmetic mean of a column."""
    col_values = [row[col_idx] for row in matrix]
    return sum(col_values) / len(col_values)


def calculate_column_median(matrix: list[list[float]], col_idx: int) -> float:
    """Compute the exact median of a column using sorting."""
    sorted_values = sorted(row[col_idx] for row in matrix)
    n = len(sorted_values)
    mid = n // 2
    if n % 2 == 1:
        return sorted_values[mid]
    return (sorted_values[mid - 1] + sorted_values[mid]) / 2.0

## 4. Verification and Benchmark Evaluation

We evaluate the metrics on `advertising_benchmark.csv` and verify the exact numerical outputs.

In [ ]:
# 1. Column Sums across TV, Radio, Newspaper, Sales
col_sums = {feat: calculate_column_sum(advertising_lst, i) for i, feat in enumerate(feature_names)}
print("--- Full Column Sums ---")
for feat, val in col_sums.items():
    print(f"  {feat.upper()} Sum: {val:.4f}")

# 2. Slice Sums: row 2 to row 11 (0-indexed indices [1:11])
col_slice_sums = {
    feat: calculate_slice_sum(advertising_lst, i, 1, 11) for i, feat in enumerate(feature_names)
}
print("\n--- Slice Sums (Row 2 to 11) ---")
for feat, val in col_slice_sums.items():
    print(f"  {feat.upper()} Slice Sum: {val:.4f}")

# 3. Min and Max
col_min_max = {
    feat: calculate_column_min_max(advertising_lst, i) for i, feat in enumerate(feature_names)
}
print("\n--- Min and Max ---")
for feat, (mn, mx) in col_min_max.items():
    print(f"  {feat.upper()} Min: {mn:.2f}, Max: {mx:.2f}")

# 4. Means and Medians
col_means = {
    feat: calculate_column_mean(advertising_lst, i) for i, feat in enumerate(feature_names)
}
col_medians = {
    feat: calculate_column_median(advertising_lst, i) for i, feat in enumerate(feature_names)
}
print("\n--- Means and Medians ---")
for feat in feature_names:
    print(f"  {feat.upper()}: Mean = {col_means[feat]:.4f}, Median = {col_medians[feat]:.4f}")

## 5. Budget Contribution Modeling

The total advertising investment comprises TV, Radio, and Newspaper expenditures. We calculate each channel's percentage contribution to the promotional mix:
$$	ext{Total Budget} = \sum 	ext{TV} + \sum 	ext{Radio} + \sum 	ext{Newspaper}$$

In [ ]:
tv_sum = col_sums["tv"]
radio_sum = col_sums["radio"]
newspaper_sum = col_sums["newspaper"]

total_promotional_budget = tv_sum + radio_sum + newspaper_sum
tv_percentage = (tv_sum / total_promotional_budget) * 100.0
radio_percentage = (radio_sum / total_promotional_budget) * 100.0
newspaper_percentage = (newspaper_sum / total_promotional_budget) * 100.0

print(f"Total Promotional Budget: ${total_promotional_budget:,.2f}k")
print(f"  TV Contribution:        {tv_percentage:.3f}%")
print(f"  Radio Contribution:     {radio_percentage:.3f}%")
print(f"  Newspaper Contribution: {newspaper_percentage:.3f}%")

# Assert mathematical consistency
assert math.isclose(tv_percentage + radio_percentage + newspaper_percentage, 100.0)
print("Budget share sums to 100.0% verified.")

## 6. Algorithmic Complexity and Vectorized Benchmarks

We analyze the algorithmic complexity of pure Python iteration vs. vectorized execution:
- Pure Python iteration has $O(N)$ time complexity and involves Python object overhead per element.
- Matrix transposition or column extraction creates intermediate list structures.
- Vectorized array operations leverage contiguous C-contiguous memory blocks and SIMD instructions.

In [ ]:
def benchmark_python_vs_custom():
    iterations = 500
    start_t = time.perf_counter()
    for _ in range(iterations):
        _ = calculate_column_mean(advertising_lst, 0)
        _ = calculate_column_median(advertising_lst, 0)
    duration = time.perf_counter() - start_t
    print(f"Pure Python {iterations} mean+median runs: {duration * 1000:.2f} ms")


benchmark_python_vs_custom()